# Lesson 11: The training loop

You now have every piece: a dataset served in batches (Lesson 8), a model that turns inputs into logits (Lesson 6), a loss that scores how wrong it is (Lesson 9), and backprop plus an optimizer that nudge the weights downhill (Lesson 10). This lesson snaps them together into the training loop and uses it to teach an MLP to recognize MNIST digits, then measures how well it did.

Run every cell top to bottom, then do the exercises at the end and upload this notebook for feedback and a grade. Training runs on CPU and takes a minute or two.

## Setup: data, model, loss, optimizer

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

torch.manual_seed(0)

train_ds = datasets.MNIST('./data', train=True,  download=True, transform=transforms.ToTensor())
test_ds  = datasets.MNIST('./data', train=False, download=True, transform=transforms.ToTensor())
train_loader = DataLoader(train_ds, batch_size=64,   shuffle=True)
test_loader  = DataLoader(test_ds,  batch_size=1000, shuffle=False)

model = nn.Sequential(nn.Linear(784, 128), nn.ReLU(), nn.Linear(128, 10))
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

## The loop, in one place

For every batch: flatten the images, then run the four training steps from Lesson 10. One pass over all the batches is an epoch.

In [ ]:
for epoch in range(3):
    model.train()
    for images, labels in train_loader:
        x = images.view(images.size(0), -1)   # flatten (1,28,28) -> 784
        optimizer.zero_grad()                 # 1. clear old gradients
        loss = criterion(model(x), labels)    # forward + how wrong
        loss.backward()                       # 2. backprop fills every .grad
        optimizer.step()                      # 3. nudge weights downhill
    print(f'epoch {epoch}: last-batch loss {loss.item():.4f}')

The loss trends down across epochs as the weights improve. That loop, forward, loss, backward, step, over batches and epochs, is the entire training algorithm.

## Measure accuracy on the held-out test set

Training loss tells you how well the model fits the training data. What you actually care about is accuracy on data it never trained on (Lesson 8). Evaluation needs no gradients, so switch to eval mode and wrap it in `torch.no_grad()`.

In [ ]:
model.eval()
correct = 0
total = 0
with torch.no_grad():
    for images, labels in test_loader:
        x = images.view(images.size(0), -1)
        preds = model(x).argmax(dim=-1)          # highest-scoring class per image
        correct += (preds == labels).sum().item()
        total += labels.size(0)

print(f'test accuracy: {correct / total:.4f}')   # around 0.97

A plain MLP reaches about 97 percent on MNIST in just a few epochs. `argmax` picks the predicted class, `(preds == labels)` is a tensor of True/False, and summing it counts the correct ones.

## train() vs eval(), and no_grad()

- `model.train()` and `model.eval()` switch the model's mode. Some layers (dropout, batch-norm) behave differently in each. Our MLP has none, so it changes nothing here, but it is a habit worth building because later models do.
- `torch.no_grad()` tells PyTorch not to build the graph for gradients during evaluation. You will not call `backward()`, so skipping it saves memory and time.

## The same loop trains everything

This exact loop, forward, loss, backward, step over batches and epochs, trains every model in this course, including the GPT. When you get there, only three things change: the data (token sequences instead of images), the model (a transformer instead of an MLP), and the target (the next token instead of the digit class). The loop itself does not change. You have now built the engine of deep learning.

## Exercises

Do these in the cells below, then upload the notebook. Only your code under each `# YOUR CODE HERE` line is graded. They reuse `train_loader` and `test_loader` from above.

In [ ]:
# E1: Wrap the loop body in a reusable function. Write train_step(model, optimizer, x, labels)
#     that runs the four steps (zero_grad, loss = F.cross_entropy(model(x), labels), backward,
#     step) and RETURNS loss.item(). Test it on model = nn.Linear(784, 10), a fake batch
#     x = torch.randn(8, 784) and labels = torch.randint(0, 10, (8,)); print the returned loss.
# YOUR CODE HERE (write your answer below)


In [ ]:
# E2: Write a reusable accuracy(logits, labels) function that returns the fraction correct
#     (argmax over the last dim, compare to labels, take the mean). Test it on
#     logits = torch.tensor([[2., 1., 0.], [0., 5., 1.], [3., 0., 1.]]) and
#     labels = torch.tensor([0, 1, 2]); it should return 2/3.
# YOUR CODE HERE (write your answer below)


In [ ]:
# E3: Swap the optimizer. Train a FRESH model = nn.Sequential(nn.Linear(784,128), nn.ReLU(),
#     nn.Linear(128,10)) for 1 epoch over train_loader, but using torch.optim.SGD(lr=0.1)
#     instead of Adam. Then compute and print its test accuracy on test_loader.
# YOUR CODE HERE (write your answer below)


In [ ]:
# E4: Overfitting check. For the model trained in the cells above, compute the accuracy on the
#     TRAIN set as well as the test set, and print both. Note whether they are close (a big
#     gap, train much higher than test, would mean the model is overfitting).
# YOUR CODE HERE (write your answer below)
